In [ ]:
import os 
import glob
import warnings
import pickle
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm

from biophysical_model.dopamine_toolbox import DA, analyze_spikes_from_file,initialize_variables_dict,udpate_results_dict
from utils.path import PathConfig

warnings.filterwarnings("ignore")
rseed =10
np.random.seed(rseed)

##  Biophysical simulations

The code below performs biophysical simulations of dopamine release and receptor occupancy, having as inputs the dopamine firing rates recorded in Tian & Uchida, 2015

These simulations are based on the  [codebase](https://github.com/jakobdreyer/Dopamine-Simulation-Tools) released by Jakob Dreyer.

The output of the models are saved as dictionaries in `pickle` files for each single neuron. 

**Notes:** 
- The output folder for saving is `'data/local/analysis/biophysical_model/single_units'` (modify if needed)
- The simulations used in the manuscript are already saved in the single neuron `pickle` files that are downloaded from the OSF repository. **It is not needed to re-run the simulations** in order to reproduce the manuscript figures.


In [15]:
groups = ['control','lesion']
paths = PathConfig()
save_dir = 'data/local/analysis/biophysical_model/single_units'
if not os.path.isdir(save_dir):
    os.makedirs(save_dir)
g_dir = paths.g_dir
max_samp = 16000 
thr_trials = 200 
window_re = [-15000, 25000]
step_dt = .01
pre_run = 1

for igroup in groups:
    print('----- processing group: ' + igroup + '-----')
    data_path = os.path.join(g_dir, 'raw_data',igroup)
    file_list = glob.glob(data_path + '/*.pickle')
    for id_unit in np.arange(len(file_list)):
        str_f = file_list[id_unit]
        with open(file_list[id_unit], 'rb') as handle:
            unit = pickle.load(handle)
        events = unit['data']['events']
        sp_times = unit['data']['responses']['spike'] 
        da = DA("vta")
        event_re = events['odorOn']
        n_trials = np.min((thr_trials,len(event_re))) 
        simulation_results = initialize_variables_dict(max_samp,n_trials)
        
        for i in tqdm(np.arange(n_trials),'Unit '+str(id_unit)+ ' of '+ str(len(file_list) )+ ' || Trial'):
            event_t = event_re[i]
            window_indeces =  np.intersect1d(np.argwhere((sp_times > (event_t + window_re[0])) ),
                                np.argwhere((sp_times < (event_t + window_re[1])) ))    
            spt = sp_times[window_indeces]
            if len(spt)>2:
                sp_times_unit = np.divide(spt,1000) 
                da = DA("vta")
                Result, Results = analyze_spikes_from_file(sp_times_unit, 
                        da, 
                        dt = step_dt, 
                        synch = 'auto', 
                        pre_run = pre_run, 
                        tmax = None, 
                        process = True, 
                        adjust_t = True, 
                        verbose=False)
                simulation_results = udpate_results_dict(Results,simulation_results,max_samp,i)
            uname = str_f.split('/')[-1].split('.pickle')[0]

        with open(os.path.join(save_dir,uname + '_biophysical.pickle'), 'wb') as handle:
            pickle.dump(simulation_results, handle, protocol=pickle.HIGHEST_PROTOCOL)

----- processing group: control-----


Unit 0 of 44 || Trial:   8%|▊         | 17/200 [00:03<00:34,  5.32it/s]